In [ ]:
import torch
import os
import cv2
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [ ]:
# data
class MyDataset(Dataset):
    def __init__(self, image_path, label_path, transform=None):
        self.image_path = image_path
        self.label_path = label_path
        self.image_files = os.listdir(image_path)
        self.len = len(self.image_files)

    def __getitem__(self, idx):
        file = self.image_files[idx]
        img_name = os.path.join(self.image_path, file)
        label_name = os.path.join(self.label_path, file.replace("jpg", "txt"))
        
        img = cv2.imread(img_name)
        img = cv2.resize(img, (64, 64))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        cls = -1
        with open(label_name, "r") as f:
            line = f.readline().strip()
            cls, x, y, w, h = list(map(float, line.split()))

        cls = torch.tensor(cls, dtype=torch.long)
        return img, cls
        

    def __len__(self):
        return self.len

In [ ]:
myds = MyDataset("./data/images", "./data/labels")
print(len(myds))

In [ ]:
import torch.nn as nn

In [158]:
# model
"""
class CatDogModel(nn.Module):
    def __init__(self):
        super(CatDogModel, self).__init__()
#        self.padding = nn.ZeroPad2d(3)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.maxpool = nn.MaxPool2d(2)      
        self.fc1 = nn.LazyLinear(1)

    def forward(self, x):
#        x = self.padding(x)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.maxpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x
"""
class CatDogModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.maxpool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(0.4)
        self.fc1 = nn.LazyLinear(1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.maxpool(x)
        x = x.flatten(1)
        x = self.drop(x)
        x = self.fc1(x)
        return x
        

In [159]:
# train
model = CatDogModel()

In [160]:
trainloader = DataLoader(myds, batch_size=4, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

In [161]:
nepoch = 10
for ep in range(nepoch):
    model.train()
    running_loss = 0
    for x, y in trainloader:
        opt.zero_grad()
        yhat = model(x)
        loss = criterion(yhat, y.unsqueeze(1).float())
        running_loss += loss.item()
        loss.backward()
        opt.step()
    print(f"epoch: {ep+1}/{nepoch}, loss: {running_loss/len(trainloader)}")

epoch: 1/10, loss: 6.006590139865875
epoch: 2/10, loss: 5.1245451927185055
epoch: 3/10, loss: 0.7913333140313625
epoch: 4/10, loss: 0.5727900199592113
epoch: 5/10, loss: 0.05113926799102728
epoch: 6/10, loss: 0.11581767272000434
epoch: 7/10, loss: 0.00020921875157000613
epoch: 8/10, loss: 0.0008706712404020322
epoch: 9/10, loss: 0.0010873776615767383
epoch: 10/10, loss: 0.002076301289469029


In [168]:
# val

test_image = cv2.imread("./data/images/dog.2.jpg")
test_image = cv2.resize(test_image, (64, 64))
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
test_image = test_image / 255.0
print(test_image.shape)

test_image_tensor = torch.tensor(test_image, dtype=torch.float32).permute(2, 0, 1)
test_image_tensor = test_image_tensor.unsqueeze(0)

(64, 64, 3)


In [169]:
model.eval()
with torch.no_grad():
    logits = model(test_image_tensor)
    prob = torch.sigmoid(logits).item()
print("logits:", logits.item())
print("prob:", prob)

logits: 14.777975082397461
prob: 0.9999996423721313
